# Day 1 - Exercise 02: Building Your First LangChain Agent

## Real-World Scenario: Personal Research Assistant

**Context:** You're building a research assistant that can help users gather information from the web, perform calculations, and answer questions with cited sources.

**Learning Objectives:**
- Understand LangChain's core components
- Build an agent from scratch
- Integrate multiple tools
- Observe the ReAct (Reason + Act) pattern in action

---

### SETUP

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain.agents import create_agent

llm = ChatAnthropic(
    api_key=os.environ.get("KEY"),
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    base_url="https://llmgw-wp.tekstac.com"
)
print("Setup complete!")

Setup complete!


---
## Part 1: Understanding the Agent Components

An agent has four core components:

| Component | Purpose | LangChain Class |
|-----------|---------|----------------|
| **LLM** | The "brain" that reasons | `ChatAnthropic` |
| **Tools** | Actions the agent can take | `@tool` decorated functions |
| **Prompt** | Instructions for the agent | `ChatPromptTemplate` |
| **Memory** | Conversation history | List of messages |

Let's build each one step by step.

### Step 1: Define Tools

Tools are Python functions that the agent can call. Each tool must have:
- A clear **name** (function name)
- A **description** (docstring) — the agent reads this to decide when to use the tool
- **Parameters** with type hints

In [2]:
import math
from datetime import datetime

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression. Supports basic operations (+, -, *, /, **) and functions like sqrt, sin, cos.
    Examples: '2 + 2', '10 ** 2', 'sqrt(16)', '100 / 4'"""
    try:
        # Safe evaluation with limited functions
        allowed_names = {
            'sqrt': math.sqrt, 'sin': math.sin, 'cos': math.cos,
            'tan': math.tan, 'log': math.log, 'pi': math.pi, 'e': math.e
        }
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def get_current_datetime() -> str:
    """Get the current date and time. Use this when the user asks about today's date or current time."""
    now = datetime.now()
    return f"Current date and time: {now.strftime('%A, %B %d, %Y at %I:%M %p')}"

@tool
def search_knowledge_base(query: str) -> str:
    """Search our internal knowledge base for information about AI, Machine Learning, and LangChain."""
    # Simulated knowledge base
    knowledge = {
        "langchain": "LangChain is an open-source framework for building LLM applications. Created by Harrison Chase in 2022. Key components: Chains, Agents, Memory, Tools.",
        "agentic ai": "Agentic AI refers to AI systems that can plan, reason, and take autonomous actions to achieve goals. Key features: tool use, memory, multi-step reasoning.",
        "claude": "Claude is an AI assistant created by Anthropic. Known for being helpful, harmless, and honest. Supports up to 200K context window.",
        "react": "ReAct (Reason + Act) is a prompting pattern where the AI thinks step-by-step, takes actions, observes results, and repeats until done.",
        "rag": "RAG (Retrieval-Augmented Generation) combines LLMs with external knowledge retrieval. Steps: Query → Retrieve documents → Generate answer with context."
    }
    
    query_lower = query.lower()
    for key, value in knowledge.items():
        if key in query_lower:
            return f"Knowledge Base Result:\n{value}"
    return "No matching information found in the knowledge base."

# Collect all tools
tools = [calculator, get_current_datetime, search_knowledge_base]
print("Tools defined:")
for t in tools:
    print(f"  - {t.name}: {t.description[:60]}...")

Tools defined:
  - calculator: Evaluate a mathematical expression. Supports basic operation...
  - get_current_datetime: Get the current date and time. Use this when the user asks a...
  - search_knowledge_base: Search our internal knowledge base for information about AI,...


### Step 2: Create the Prompt Template

The prompt tells the agent:
- Who it is (system message)
- What it can do (tools are automatically injected)
- How to behave

In [3]:
system_prompt = """You are a helpful research assistant with access to tools.

Your capabilities:
- Calculate mathematical expressions
- Get the current date and time
- Search our knowledge base about AI topics

Guidelines:
1. Always use tools when you need factual information
2. Think step-by-step before answering complex questions
3. Cite your sources when using the knowledge base
4. If you don't know something, say so honestly
"""

print("System prompt created!")

System prompt created!


### Step 3: Assemble the Agent

Now we combine the LLM, tools, and prompt into an agent.

In [4]:
# Create the agent using LangChain 1.x API
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

print("Agent created!")

Agent created!


### Step 4: Add Memory

Memory is simply a list of messages that gets passed to the agent each time.

In [5]:
chat_history = []  # Our memory store

def chat(user_input: str) -> str:
    """Send a message to the agent and update memory."""
    global chat_history
    
    # Build messages list: chat history + current input
    messages = chat_history + [HumanMessage(content=user_input)]
    
    # Invoke the agent with messages
    result = agent.invoke({"messages": messages})
    
    # Extract the response (it's the last message in the result)
    response_text = result["messages"][-1].content
    
    # Update memory
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response_text))
    
    return response_text

print("Chat function ready!")

Chat function ready!


---
## Part 2: Testing the Agent

Let's see the agent in action with different types of questions.

In [6]:
# Test 1: Calculator tool
print("="*60)
print("Test 1: Math calculation")
print("="*60)
response = chat("What is 15% of 250?")
print(f"\nFinal Answer: {response}")

Test 1: Math calculation



Final Answer: 15% of 250 is **37.5**.


In [7]:
# Test 2: Knowledge base search
print("="*60)
print("Test 2: Knowledge lookup")
print("="*60)
response = chat("What is LangChain and who created it?")
print(f"\nFinal Answer: {response}")

Test 2: Knowledge lookup

Final Answer: Based on our knowledge base:

**LangChain** is an open-source framework for building Large Language Model (LLM) applications. It was created by **Harrison Chase in 2022**.

The framework provides several key components that make it easier to develop LLM-powered applications:
- **Chains** - Sequences of operations for processing data
- **Agents** - Autonomous systems that can make decisions and take actions
- **Memory** - Mechanisms for storing and retrieving information
- **Tools** - Integrations with external services and APIs

LangChain has become a popular framework in the AI development community for building sophisticated applications that leverage large language models.


In [8]:
# Test 3: Multi-tool usage
print("="*60)
print("Test 3: Complex question requiring multiple tools")
print("="*60)
response = chat("What is today's date, and if LangChain was created in 2022, how many years ago was that?")
print(f"\nFinal Answer: {response}")

Test 3: Complex question requiring multiple tools

Final Answer: Today's date is **Wednesday, July 1, 2026** at 3:05 PM.

If LangChain was created in 2022, that was **4 years ago** (2026 - 2022 = 4 years).


In [9]:
# Test 4: Memory test
print("="*60)
print("Test 4: Memory - follow-up question")
print("="*60)
response = chat("Based on what you told me earlier, what are the key components of LangChain?")
print(f"\nFinal Answer: {response}")

Test 4: Memory - follow-up question

Final Answer: Based on what I told you earlier, the key components of LangChain are:

1. **Chains** - Sequences of operations for processing data
2. **Agents** - Autonomous systems that can make decisions and take actions
3. **Memory** - Mechanisms for storing and retrieving information
4. **Tools** - Integrations with external services and APIs

These components work together to make it easier to develop sophisticated applications powered by large language models.


---
## Part 3: Observing the ReAct Pattern

The `verbose=True` setting shows us the agent's thinking process:

```
Thought: I need to calculate 15% of 250
Action: calculator
Action Input: "250 * 0.15"
Observation: Result: 37.5
Thought: I now have the answer
Final Answer: 15% of 250 is 37.5
```

This is the **ReAct loop**: Reason → Act → Observe → Repeat

---
##  Your Turn: Practice Exercises

### Exercise 2.1: Add a New Tool

Create a tool that converts between units (temperature, length, weight).

In [10]:
 
@tool
def convert_units(value: float, from_unit: str, to_unit: str) -> str:
    """Convert between units. Supports:
    - Temperature: celsius, fahrenheit, kelvin
    - Length: meters, feet, inches, kilometers, miles
    - Weight: kg, pounds, ounces
    """
    # Normalize inputs to avoid case sensitivity or trailing spaces
    from_unit = from_unit.lower().strip()
    to_unit = to_unit.lower().strip()
    
    if from_unit == to_unit:
        return f"{value} {from_unit} is equal to {value} {to_unit}"
 
    # 1. Temperature Conversion
    if from_unit in ['celsius', 'fahrenheit', 'kelvin']:
        if to_unit not in ['celsius', 'fahrenheit', 'kelvin']:
            return f"Error: Cannot convert temperature unit '{from_unit}' to '{to_unit}'."
        
        # Convert input to base unit (Celsius)
        if from_unit == 'celsius':
            celsius_val = value
        elif from_unit == 'fahrenheit':
            celsius_val = (value - 32) * 5 / 9
        elif from_unit == 'kelvin':
            celsius_val = value - 273.15
            
        # Convert from Celsius to target unit
        if to_unit == 'celsius':
            result = celsius_val
        elif to_unit == 'fahrenheit':
            result = (celsius_val * 9 / 5) + 32
        elif to_unit == 'kelvin':
            result = celsius_val + 273.15
            
        return f"{value} {from_unit} is equal to {result:.2f} {to_unit}"
 
    # 2. Length Conversion
    elif from_unit in ['meters', 'feet', 'inches', 'kilometers', 'miles']:
        if to_unit not in ['meters', 'feet', 'inches', 'kilometers', 'miles']:
            return f"Error: Cannot convert length unit '{from_unit}' to '{to_unit}'."
        
        # Conversion factors to base unit (meters)
        meters_factors = {
            'meters': 1.0,
            'kilometers': 1000.0,
            'feet': 0.3048,       # Aligns with meters to feet: value * 3.28084
            'inches': 0.0254,
            'miles': 1609.344
        }
        
        meters_val = value * meters_factors[from_unit]
        result = meters_val / meters_factors[to_unit]
        
        return f"{value} {from_unit} is equal to {result:.4f} {to_unit}"
 
    # 3. Weight Conversion
    elif from_unit in ['kg', 'pounds', 'ounces']:
        if to_unit not in ['kg', 'pounds', 'ounces']:
            return f"Error: Cannot convert weight unit '{from_unit}' to '{to_unit}'."
        
        # Conversion factors to base unit (kg)
        kg_factors = {
            'kg': 1.0,
            'pounds': 0.45359237,  # Aligns with kg to pounds: value * 2.20462
            'ounces': 0.45359237 / 16
        }
        
        kg_val = value * kg_factors[from_unit]
        result = kg_val / kg_factors[to_unit]
        
        return f"{value} {from_unit} is equal to {result:.4f} {to_unit}"
        
    else:
        return f"Error: Unsupported unit format '{from_unit}'."
 

In [11]:
# Test your tool
result = convert_units.invoke({"value": 100, "from_unit": "celsius", "to_unit": "fahrenheit"})
print(result)

100.0 celsius is equal to 212.00 fahrenheit


### Exercise 2.2: Customize the Agent Persona

Modify the system prompt to create a **Financial Advisor** agent that:
- Uses formal language
- Always provides disclaimers
- Focuses on financial calculations and advice

In [19]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_anthropic import ChatAnthropic
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
 
# =====================================================================
# 1. LOAD LAB GATEWAY ROUTING CONFIGURATION
# =====================================================================
load_dotenv()
 
# Initialize the native Anthropic client pointed at your lab gateway URL
llm = ChatAnthropic(
    anthropic_api_url=os.getenv("BASE_URL"),
    anthropic_api_key=os.getenv("KEY"),
    model_name=os.getenv("MODEL"),
    temperature=0
)
 
# =====================================================================
# 2. DEFINE SYSTEM PROMPT & GUARDRAILS
# =====================================================================
financial_advisor_prompt_str = """You are a professional, elite financial advisor assistant. 
 
### PERSONA:
- Maintain an authoritative, objective, and highly formal tone in all interactions. 
- Use precise financial and economic terminology where appropriate. Avoid casual language, slang, or overly familiar phrasing.
 
### GUIDELINES FOR RESPONSES:
- Focus exclusively on financial advice, wealth management, asset allocation, budgeting strategies, and financial calculations.
- When performing financial math, break down the steps clearly so the logic is transparent and verifiable.
- If a query falls completely outside the scope of finance, economics, or fiscal planning, politely decline to answer, redirecting the user back to financial matters.
 
### DISCLAIMER REQUIREMENTS:
- You MUST append a clear, standardized disclaimer to the conclusion of every response.
- The disclaimer must state that the information provided is for educational and illustrative purposes only and should not be treated as licensed fiduciary, legal, or tax advice."""
 
# =====================================================================
# 3. INITIALIZE AGENT INFRASTRUCTURE
# =====================================================================
 
# Bind the unit converter tool defined in Exercise 2.1
tools = [convert_units]
 
# Assemble the prompt layout
prompt = ChatPromptTemplate.from_messages([
    ("system", financial_advisor_prompt_str),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])
 
# Construct the classic tool-calling agent using the Anthropic instance
agent = create_tool_calling_agent(llm, tools, prompt)
 
# Build the execution engine container
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
 
print("Financial Advisor Agent successfully initialized using native Anthropic routing!")

Financial Advisor Agent successfully initialized using native Anthropic routing!


In [20]:
# Test Case: Financial Calculation with Persona Guidelines
print("--- RUNNING PILOT CONTEXT VALIDATION ---")
test_financial = agent_executor.invoke({
    "input": "If I invest $10,000 with a 5% annual interest rate compounded annually, what is the balance after 3 years?"
})
print("\n[Agent Response]:\n", test_financial["output"])

--- RUNNING PILOT CONTEXT VALIDATION ---


> Entering new AgentExecutor chain...
[{'text': 'I shall calculate the future value of your investment using the compound interest formula.\n\n**Calculation:**\n\nThe compound interest formula is:\n$$A = P(1 + r)^t$$\n\nWhere:\n- **P** = Principal (initial investment) = $10,000\n- **r** = Annual interest rate (decimal form) = 0.05\n- **t** = Time period in years = 3\n- **A** = Final amount\n\n**Step-by-step computation:**\n\n$$A = 10,000(1 + 0.05)^3$$\n\n$$A = 10,000(1.05)^3$$\n\n$$A = 10,000 × 1.157625$$\n\n$$A = \\$11,576.25$$\n\n**Result:**\n\nAfter 3 years, your investment balance will be **$11,576.25**, representing a total gain of **$1,576.25** in interest earnings.\n\nThis assumes:\n- No additional deposits or withdrawals during the period\n- Interest is compounded annually (not more frequently)\n- The stated rate remains constant throughout the investment horizon\n\n---\n\n**DISCLAIMER:** This information is provided for educational an

### Exercise 2.3: Build a Complete Agent

Build a **Travel Planning Agent** with these tools:
1. `get_weather(city)` — Returns weather for a city
2. `search_flights(from_city, to_city, date)` — Searches for flights
3. `get_attractions(city)` — Lists popular attractions

Use simulated data for the tools.

In [21]:
 
# =====================================================================
# Step 1: Define simulated data
# =====================================================================
WEATHER_DATA = {
    "paris": "Sunny, 22°C",
    "tokyo": "Cloudy, 18°C",
    "new york": "Rainy, 15°C"
}
 
ATTRACTIONS = {
    "paris": ["Eiffel Tower", "Louvre Museum", "Notre-Dame"],
    "tokyo": ["Tokyo Tower", "Senso-ji Temple", "Shibuya Crossing"],
    "new york": ["Statue of Liberty", "Central Park", "Times Square"]
}
 
# =====================================================================
# Step 2: Define tools
# =====================================================================
@tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    city_clean = city.lower().strip()
    return WEATHER_DATA.get(city_clean, f"Weather data not found for {city}.")
 
@tool
def search_flights(from_city: str, to_city: str, date: str) -> str:
    """Search for available flights."""
    return f"Simulated Flight: Found a flight from {from_city.strip()} to {to_city.strip()} on {date.strip()} for $450."
 
@tool
def get_attractions(city: str) -> str:
    """Get popular tourist attractions in a city."""
    city_clean = city.lower().strip()
    if city_clean in ATTRACTIONS:
        return f"Popular spots in {city}: " + ", ".join(ATTRACTIONS[city_clean])
    return f"No attraction recommendations found for {city}."
 
print("Simulated travel data and tools configured successfully.")
 

Simulated travel data and tools configured successfully.


In [22]:
# Load existing environment credentials
load_dotenv()
 
# Initialize the gateway-bound model client
llm = ChatAnthropic(
    anthropic_api_url=os.getenv("BASE_URL"),
    anthropic_api_key=os.getenv("KEY"),
    model_name=os.getenv("MODEL"),
    temperature=0
)
 
# Compile tools list
travel_tools = [get_weather, search_flights, get_attractions]
 
# =====================================================================
# Step 3: Create the agent
# =====================================================================
travel_agent_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert personal travel planning assistant.
    Your goal is to help users plan trips efficiently by checking weather conditions,
    finding flights, and suggesting top local tourist attractions.
    Always present information clearly and enthusiastically."""),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])
 
# Build the execution engine container
agent = create_tool_calling_agent(llm, travel_tools, travel_agent_prompt)
travel_agent_executor = AgentExecutor(agent=agent, tools=travel_tools, verbose=True)
 
print("Travel Planning Agent successfully initialized!")

Travel Planning Agent successfully initialized!


In [23]:
# =====================================================================
# Step 4: Test with specific travel prompt
# =====================================================================
test_input = "I'm planning a trip to Paris next week. What's the weather like and what should I see?"
 
print("--- RUNNING TRAVEL AGENT EXECUTION LOOP ---")
response = travel_agent_executor.invoke({"input": test_input})
 
print("\n[Final Agent Response]:\n", response["output"])
 

--- RUNNING TRAVEL AGENT EXECUTION LOOP ---


> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'city': 'Paris'}`
responded: [{'text': "I'd be happy to help you plan your Paris trip! Let me check the weather and get you some great tourist attractions to visit.", 'type': 'text', 'index': 0}, {'id': 'toolu_bdrk_01Kzp2NV3cceq9zx8uNufV8B', 'input': {}, 'name': 'get_weather', 'type': 'tool_use', 'index': 1, 'partial_json': '{"city": "Paris"}'}, {'id': 'toolu_bdrk_015Jsrfj3oubXK41YzftPpM2', 'input': {}, 'name': 'get_attractions', 'type': 'tool_use', 'index': 2, 'partial_json': '{"city": "Paris"}'}]

Sunny, 22°C
Invoking: `get_attractions` with `{'city': 'Paris'}`
responded: [{'text': "I'd be happy to help you plan your Paris trip! Let me check the weather and get you some great tourist attractions to visit.", 'type': 'text', 'index': 0}, {'id': 'toolu_bdrk_01Kzp2NV3cceq9zx8uNufV8B', 'input': {}, 'name': 'get_weather', 'type': 'tool_use', 'index': 1, 'partial_json': '{"cit

---
##  Key Takeaways

1. **LangChain agents** combine LLM + Tools + Prompt + Memory
2. **Tools** are Python functions with clear descriptions — the agent reads docstrings to decide when to use them
3. **The prompt** defines the agent's persona and behavior
4. **Memory** is a list of messages passed to each agent call
5. **AgentExecutor** runs the ReAct loop automatically
6. **verbose=True** shows the agent's reasoning — essential for debugging

---

**Next Exercise →** Agent Workflows and Real-World Use Cases